# CS156 · Session 7 — Feed-Forward Neural Networks and Perceptrons

**Based on the supplied neural-networks PCW (7-page Forum capture) and the uploaded `07_feed_forward_nn.ipynb`.** This notebook retains the existing MNIST architecture and evaluation, reconstructs the PCW's conceptual questions and matrix multiplication, implements its single-unit Sonar perceptron, and finishes with the Fashion-MNIST extension.

**Run top to bottom.** The MNIST and Fashion-MNIST cells use TensorFlow's built-in dataset loader and download the files once when not already cached. The Sonar section uses the local CSV if available or downloads it to a user cache. Dataset training is **not** run automatically when you download this notebook.

**Session objectives:** describe layers and units; label a dense network with correct matrix dimensions; compare an MLP with a single perceptron; implement the perceptron update from scratch; investigate optional PCA on clothing images.

## 1. PCW Questions 1–2 — MNIST: a feed-forward neural network

The supplied PCW examines a network for the **MNIST handwritten digits** dataset. Input images are $28\times28$ grayscale arrays, flattened to 784 numbers. Its architecture is

$$\boxed{784\ \longrightarrow\ 128\ \longrightarrow\ 64\ \longrightarrow\ 10}$$

- **Input:** 784 image pixel values (no learned input weights).
- **Hidden layer 1:** 128 sigmoid units.
- **Hidden layer 2:** 64 sigmoid units.
- **Output:** 10 class scores followed by **softmax** in the revised code supplied with the PCW. These are the probabilities of classes 0–9.

A **unit/neuron** computes $a_j=\sigma(\sum_i w_{ji}x_i+b_j)$. Each `Dense` layer is fully connected to the previous layer; a connection weight tells us how much one input contributes to a neuron's pre-activation.

Keras normally initializes dense-layer weights using **Glorot uniform** and biases to zero. Backpropagation calculates loss gradients; SGD updates the weights. More depth means more successive learned transformations, but it also increases compute/memory use and can complicate optimization.

### Question 2: architecture and forward pass (column-vector convention)

$$
\begin{aligned}
x&\in\mathbb R^{784}, \\
h_1 &= \sigma(W_1x+b_1), & W_1&\in\mathbb R^{128\times784},\quad b_1\in\mathbb R^{128},\\
h_2 &= \sigma(W_2h_1+b_2), & W_2&\in\mathbb R^{64\times128},\quad b_2\in\mathbb R^{64},\\
z &= W_3h_2+b_3, & W_3&\in\mathbb R^{10\times64},\quad b_3\in\mathbb R^{10},\\
p_k&=\operatorname{softmax}(z)_k=\frac{e^{z_k}}{\sum_{j=0}^{9}e^{z_j}}.
\end{aligned}
$$

For a single example, $W_1$ has shape $(128,784)$ in this notation. Internally, Keras stores its kernel transposed as `(784, 128)` because it multiplies a **batch of row vectors** by the kernel.

The original `Dense(10)` example without softmax returns **logits**, not a probability mass function. The corrected PCW code adds `activation="softmax"` and uses `sparse_categorical_crossentropy` for the 10 exclusive classes.

In [ ]:
import time
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, classification_report, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score
)

RANDOM_STATE = 42

### Draw the network with labeled connections

This code recreates the PCW's 784 → 128 → 64 → 10 drawing as **four rectangles**, as the assignment permits. Biases are added before the activation in each trainable layer.

In [ ]:
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.set_xlim(0, 11); ax.set_ylim(0, 3); ax.axis("off")
boxes = [
    (0.25, "Input\n784 pixels"),
    (3.00, "Hidden 1\n128 sigmoid units"),
    (5.75, "Hidden 2\n64 sigmoid units"),
    (8.50, "Output\n10 softmax units")
]
for left, title in boxes:
    ax.add_patch(FancyBboxPatch((left, 1.05), 2.1, 1.05,
                                boxstyle="round,pad=0.08", fill=False, lw=1.6))
    ax.text(left+1.05, 1.58, title,
            ha="center", va="center", fontsize=11)
for left, label in zip([2.35, 5.10, 7.85],
                       ["W₁: 128×784; b₁: 128", "W₂: 64×128; b₂: 64",
                        "W₃: 10×64; b₃: 10"]):
    ax.annotate("", xy=(left+0.64, 1.58), xytext=(left, 1.58),
                arrowprops={"arrowstyle": "->", "lw": 1.6})
    ax.text(left+0.31, 0.55, label.replace("; ", "\n"),
            ha="center", va="center", fontsize=9)
ax.set_title("MNIST MLP: fully connected feed-forward network", pad=10)
plt.tight_layout(); plt.show()

### Train and evaluate the user's revised MNIST model

The uploaded existing Session 7 notebook already includes a 90/10 stratified validation split, two sigmoid hidden layers, SGD, test-set evaluation, learning curves and a confusion matrix. The following cells preserve those choices while removing repetitive dataset-size and class-count messages. Training logs are suppressed (`verbose=0`), and all results appear **once** at the end.

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist, fashion_mnist
from tensorflow.keras.layers import Flatten, Dense

tf.keras.utils.set_random_seed(RANDOM_STATE)

(mnist_images, mnist_labels), (mnist_test_images, mnist_y_test) = mnist.load_data()
mnist_X_train, mnist_X_val, mnist_y_train, mnist_y_val = train_test_split(
    mnist_images, mnist_labels, test_size=0.10, stratify=mnist_labels,
    random_state=RANDOM_STATE
)
mnist_X_train = mnist_X_train.astype("float32") / 255.0
mnist_X_val = mnist_X_val.astype("float32") / 255.0
mnist_X_test = mnist_test_images.astype("float32") / 255.0

mnist_model = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28)),
    Flatten(),
    Dense(128, activation="sigmoid"),
    Dense(64, activation="sigmoid"),
    Dense(10, activation="softmax")
])
mnist_model.compile(optimizer="SGD", loss="sparse_categorical_crossentropy",
                    metrics=["accuracy"])

start = time.perf_counter()
mnist_history = mnist_model.fit(
    mnist_X_train, mnist_y_train, epochs=10, batch_size=128,
    validation_data=(mnist_X_val, mnist_y_val), shuffle=True, verbose=0
)
mnist_train_time = time.perf_counter() - start

In [ ]:
mnist_train_stats = mnist_model.evaluate(
    mnist_X_train, mnist_y_train, verbose=0, return_dict=True)
mnist_val_stats = mnist_model.evaluate(
    mnist_X_val, mnist_y_val, verbose=0, return_dict=True)
mnist_test_stats = mnist_model.evaluate(
    mnist_X_test, mnist_y_test, verbose=0, return_dict=True)
mnist_pred = np.argmax(mnist_model.predict(mnist_X_test, verbose=0), axis=1)
print("MNIST | 784 → 128 → 64 → 10 |", f"parameters: {mnist_model.count_params():,}")
print(f"Train: {mnist_train_stats['accuracy']:.2%} | "
      f"Validation: {mnist_val_stats['accuracy']:.2%} | "
      f"Test: {mnist_test_stats['accuracy']:.2%} | "
      f"Time: {mnist_train_time:.1f}s")
print("Macro precision / recall / F1:",
      f"{precision_score(mnist_y_test, mnist_pred, average='macro'):.3f} / "
      f"{recall_score(mnist_y_test, mnist_pred, average='macro'):.3f} / "
      f"{f1_score(mnist_y_test, mnist_pred, average='macro'):.3f}")
print("\nPer-digit test report:\n", classification_report(mnist_y_test, mnist_pred,
      target_names=[str(i) for i in range(10)], zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for metric, title, ax in [("accuracy", "Accuracy", axes[0]),
                          ("loss", "Cross-entropy loss", axes[1])]:
    ax.plot(mnist_history.history[metric], label="Train")
    ax.plot(mnist_history.history[f"val_{metric}"], label="Validation")
    ax.set(xlabel="Epoch", ylabel=title, title=f"MNIST: {title}")
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

ConfusionMatrixDisplay.from_predictions(
    mnist_y_test, mnist_pred, display_labels=np.arange(10),
    cmap="Blues", values_format="d")
plt.title("MNIST held-out test set"); plt.tight_layout(); plt.show()

## 2. PCW Question 3 — Single-layer perceptron on Sonar

The second PCW exercise (adapted from Jason Brownlee, *Perceptron Algorithm From Scratch*) uses **208** sonar measurements with **60 numeric features** and a rock (`R`) or metal cylinder / mine (`M`) label.

- **Architecture:** 60 input features → one output neuron, no hidden layers.
- **Activation:** the Heaviside threshold $H(z)=\mathbb{1}[z\ge0]$.
- **Initial weights:** all 60 weights **and the bias** start at zero.
- **Learning rule:** for each example, $e=y-\hat y$, then $b\leftarrow b+\eta e$ and $w\leftarrow w+\eta e x$.
- **Output:** one binary class prediction per observation.
- `predict()` applies the learned weights to one observation; `train_weights()` repeatedly updates the weights for misclassified observations. This is the **perceptron update**, not differentiating through a Heaviside function.

**Small fixes to the PCW's original code:** we use a stable label convention `R=0, M=1`; cast labels to signed integers to prevent unsigned underflow; and use stratified 3-fold CV so all 208 rows are evaluated once. The PCW's original `set` label mapping may vary between runs, and its integer fold size leaves one row out.

### PCW Question 4 — Draw the Sonar perceptron and show its forward pass

$$
x=\begin{bmatrix}x_1&\cdots&x_{60}\end{bmatrix}^{\!T}\in\mathbb R^{60},\quad
W=\begin{bmatrix}w_1&\cdots&w_{60}\end{bmatrix}\in\mathbb R^{1\times60}.
$$

$$
z=Wx+b
=\begin{bmatrix}w_1&\cdots&w_{60}\end{bmatrix}
\begin{bmatrix}x_1\\\vdots\\x_{60}\end{bmatrix}+b,
\qquad \widehat y=H(z)=\begin{cases}1 & z\ge 0\\0 & z<0.\end{cases}
$$

The model has **61 trainable parameters** (60 connection weights plus one bias).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2.7))
ax.set_xlim(0, 9); ax.set_ylim(0, 3); ax.axis("off")
ax.add_patch(FancyBboxPatch((0.5, 0.95), 2.2, 1.25,
                            boxstyle="round,pad=0.1", fill=False, lw=1.8))
ax.add_patch(FancyBboxPatch((6.25, 0.95), 2.2, 1.25,
                            boxstyle="round,pad=0.1", fill=False, lw=1.8))
ax.text(1.6, 1.58, "60 sonar features\n(x₁, …, x₆₀)", ha="center", va="center")
ax.text(7.35, 1.58, "1 output unit\nHeaviside H(z)", ha="center", va="center")
ax.annotate("", xy=(6.25, 1.58), xytext=(2.7, 1.58),
            arrowprops={"arrowstyle":"->", "lw":1.8})
ax.text(4.5, 1.85, "W: 1 × 60; bias b: scalar", ha="center", fontsize=11)
ax.text(4.5, 1.05, "z = Wx + b", ha="center", fontsize=12)
ax.set_title("Single-layer perceptron: rock (0) vs. metal (1)")
plt.tight_layout(); plt.show()

In [ ]:
# Read an existing local CSV, or download Brownlee's copy to a user cache.
# This avoids assuming which directory VS Code chose as the notebook's cwd.
sonar_candidates = [
    Path("sonar.all-data.csv"),
    Path("../sonar.all-data.csv"),
    Path("data/sonar.all-data.csv"),
    Path.home() / ".cache" / "cs156" / "sonar.all-data.csv"
]
sonar_path = next((p for p in sonar_candidates if p.exists()), None)
if sonar_path is None:
    sonar_path = sonar_candidates[-1]
    sonar_path.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve("https://raw.githubusercontent.com/jbrownlee/Datasets/master/sonar.csv",
                sonar_path)

raw_sonar = np.genfromtxt(sonar_path, delimiter=",", dtype=str)
sonar_X = raw_sonar[:, :60].astype(np.float64)
sonar_y = np.array([{"R":0, "M":1}[label.strip()] for label in raw_sonar[:, 60]],
                   dtype=np.int64)
assert sonar_X.shape == (208, 60) and set(sonar_y) == {0, 1}

In [ ]:
# The perceptron learning rule is implemented from scratch.
def predict(row, weights):
    z = float(np.dot(row, weights[1:]) + weights[0])
    return int(z >= 0.0)


def predict_batch(X, weights):
    return (np.asarray(X) @ weights[1:] + weights[0] >= 0.0).astype(int)


def train_weights(X, y, l_rate=0.01, n_epoch=500, seed=42,
                  validation=None):
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.int64)  # essential: 0-1 must equal -1, not 255
    weights = np.zeros(X.shape[1] + 1, dtype=np.float64)
    rng = np.random.default_rng(seed)
    history = {"training": [], "validation": []}
    for epoch in range(n_epoch):
        for idx in rng.permutation(len(X)):
            error = int(y[idx]) - predict(X[idx], weights)
            weights[0] += l_rate * error
            weights[1:] += l_rate * error * X[idx]
        history["training"].append(accuracy_score(y, predict_batch(X, weights)))
        if validation is not None:
            val_X, val_y = validation
            history["validation"].append(
                accuracy_score(val_y, predict_batch(val_X, weights)))
    return weights, history


def perceptron(train_X, train_y, test_X, l_rate=0.01, n_epoch=500, seed=42):
    learned_weights, _ = train_weights(train_X, train_y, l_rate, n_epoch, seed)
    return predict_batch(test_X, learned_weights)


# Three stratified folds: every one of the 208 examples is tested exactly once.
sonar_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
sonar_scores = []
for fold_id, (training, testing) in enumerate(sonar_cv.split(sonar_X, sonar_y), 1):
    fold_pred = perceptron(sonar_X[training], sonar_y[training], sonar_X[testing],
                           l_rate=0.01, n_epoch=500, seed=RANDOM_STATE+fold_id)
    sonar_scores.append(accuracy_score(sonar_y[testing], fold_pred))
print("Sonar 3-fold accuracies:", ", ".join(f"{s:.1%}" for s in sonar_scores))
print(f"Mean cross-validation accuracy: {np.mean(sonar_scores):.1%}")

## 3. PCW Question 5 — Fashion-MNIST extension

**Goal:** adapt the from-scratch perceptron to distinguish *T-shirt/top* (**label 0**) from *trouser* (**label 1**). We exclude Fashion-MNIST's other eight categories, normalize and flatten images, then optionally fit PCA **on training images only**.

With PCA on, the architecture is **50 principal-component inputs → one threshold unit** (51 trainable parameters). Without PCA it is **784 pixel inputs → one threshold unit** (785 parameters). PCA is linear, so it does **not** make the perceptron's decision boundary nonlinear in the original image pixels.

Unlike the original extension's verbose output, this version prints **one final summary** and shows only learning curves and a confusion matrix. The integer cast in `train_weights()` avoids `uint8` subtraction-overflow warnings.

In [ ]:
(fashion_images, fashion_labels), (fashion_test_images, fashion_test_labels) = (
    fashion_mnist.load_data()
)
fashion_train_mask = np.isin(fashion_labels, [0, 1])
fashion_test_mask = np.isin(fashion_test_labels, [0, 1])

fashion_X = fashion_images[fashion_train_mask].reshape(-1, 784).astype("float32") / 255
fashion_y = fashion_labels[fashion_train_mask].astype(np.int64)
fashion_X_test = fashion_test_images[fashion_test_mask].reshape(-1, 784).astype("float32") / 255
fashion_y_test = fashion_test_labels[fashion_test_mask].astype(np.int64)

fashion_X_train, fashion_X_val, fashion_y_train, fashion_y_val = train_test_split(
    fashion_X, fashion_y, test_size=0.10, stratify=fashion_y,
    random_state=RANDOM_STATE
)

USE_PCA = True
N_COMPONENTS = 50
if USE_PCA:
    fashion_pca = PCA(n_components=N_COMPONENTS, svd_solver="randomized",
                      random_state=RANDOM_STATE)
    fashion_X_train = fashion_pca.fit_transform(fashion_X_train)
    fashion_X_val = fashion_pca.transform(fashion_X_val)
    fashion_X_test = fashion_pca.transform(fashion_X_test)

start = time.perf_counter()
fashion_weights, fashion_history = train_weights(
    fashion_X_train, fashion_y_train, l_rate=0.01, n_epoch=30,
    seed=RANDOM_STATE, validation=(fashion_X_val, fashion_y_val)
)
fashion_time = time.perf_counter() - start
fashion_pred = predict_batch(fashion_X_test, fashion_weights)

print(f"Fashion-MNIST | {'PCA '+str(N_COMPONENTS) if USE_PCA else '784 pixels'} | "
      f"parameters: {len(fashion_weights)} | training: {fashion_time:.1f}s")
print(f"Training: {fashion_history['training'][-1]:.2%} | "
      f"Validation: {fashion_history['validation'][-1]:.2%} | "
      f"Test: {accuracy_score(fashion_y_test, fashion_pred):.2%}")
if USE_PCA:
    print(f"PCA variance explained: {fashion_pca.explained_variance_ratio_.sum():.1%}")
print(classification_report(fashion_y_test, fashion_pred,
                            target_names=["T-shirt/top", "Trouser"], zero_division=0))

plt.figure(figsize=(7, 4))
plt.plot(fashion_history["training"], label="Training")
plt.plot(fashion_history["validation"], label="Validation")
plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.title("Fashion-MNIST perceptron learning curves")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

ConfusionMatrixDisplay.from_predictions(fashion_y_test, fashion_pred,
    display_labels=["T-shirt/top", "Trouser"], cmap="Blues", values_format="d")
plt.title("Fashion-MNIST held-out test set")
plt.tight_layout(); plt.show()

### Extension reflection

1. Set `USE_PCA=False` and rerun the **Fashion-MNIST section** (restart the kernel first, or rerun its data-loading cell so that you start with 784-pixel features). Compare test accuracy and training time.
2. Why does PCA need to be `fit` on **training data only**, rather than the combined training and test sets?
3. Why can the single perceptron separate only linearly separable patterns even after PCA?
4. How do this model's **51 (or 785) trainable parameters** compare with the MNIST MLP's parameter count?

*Sources: supplied 7-page neural-networks PCW, pp. 1–7, and the uploaded user's MNIST notebook. The labeled diagrams, deterministic Sonar folds, local CSV fallback, quiet training output and explicit evaluation are notebook-editing additions.*